# QC Pipeline — African-American Cohort (GSE148375)

**Purpose:** Clean raw genotype calls and produce a QC-passed, encoded 0/1/2 genotype
matrix ready for confounder adjustment and causal analysis.

**Inputs:**
- `checkpoint0_snp_clean_v2.txt` — raw genotype calls (NC = no call), indels/ex-smokers already removed
- `checkpoint1_metadata_binary.csv` — sample metadata (smoking_status, age, gender, cpd)
- `HumanExome-12-v1-0-B.csv` — Illumina manifest (chromosome, position, allele info)

**Key parameters:**

| Parameter | Value | Rationale |
|---|---|---|
| Sample missingness threshold | 2% | drops 48 samples |
| Probe missingness threshold | 5% | drops 3,837 probes |
| HWE flag threshold | p < 1e-6 | flagged only, not dropped (per supervisor) |
| MAF filter | none (genome-wide) | per supervisor; phenotype-specific subsamples (e.g. CPD smoker-only) get their own MAF floor separately, see 03_snp_selection notebooks |
| Imputation | per-probe mode | fills NC with most common observed genotype at that probe |
| Reference allele | self-computed major allele | NOT Illumina manifest strand — see Step 6 note |
| Relatedness threshold | pairwise correlation > 0.5 | drops duplicate/close-relative samples — see Step 8 |

**Outputs:**
- `checkpoint7b_snp_encoded_012_relatedness_filtered.csv` — final encoded matrix (probes x samples, 0/1/2), relatedness-filtered
- `checkpoint2b_metadata_relatedness_filtered.csv` — metadata aligned to final surviving samples
- QC logs: dropped samples, dropped probes, HWE-flagged probes, dropped-for-relatedness samples

**Important methodological note — why major allele, not manifest strand:**
An earlier version of this pipeline attempted to encode genotypes using the Illumina
manifest's `RefStrand`/`IlmnStrand`/`TopGenomicSeq` fields to determine each probe's
reference allele on a consistent forward strand. This produced encoding errors
(known monomorphic test probes came out non-monomorphic). The pipeline instead
computes the major allele directly from the observed genotype data for each probe,
which is strand-agnostic by construction and was validated against known
monomorphic test probes. The abandoned strand-based attempt is preserved in
`archive/AA_qc_strand_investigation_deadend.ipynb` for reference.

**Important methodological note — why relatedness filtering (Step 8):**
Discovered via the genomic inflation factor (lambda) diagnostic in `02_confounders.ipynb`,
which came back abnormally high (lambda ~1.7-1.8 at 10 PCs) and got WORSE with more PCs —
the signature of relatedness contamination, not uncorrected population structure
(population structure is smooth/continuous and more PCs should capture it; relatedness
is discrete/local and PCs cannot fix it). Pairwise genotype correlation revealed ~9%
of the cohort were duplicate/closely-related samples. This step removes them before
any downstream analysis.

## Step 1 — Sample missingness filter

Compute per-sample NC (no-call) rate across all probes. Samples above threshold
are dropped entirely (both from genotypes and metadata) before any probe-level QC,
since a bad sample inflates probe missingness for every probe it touches.

**Result:** 48 samples dropped (3,396 -> 3,348 samples), threshold = 2% missingness.

In [ ]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint0_snp_clean_v2.txt"

chunksize = 20000
sample_nc_counts = None
sample_cols = None
total_probes = 0

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    if sample_cols is None:
        sample_cols = chunk.columns[1:]
        sample_nc_counts = pd.Series(0, index=sample_cols)

    total_probes += len(chunk)
    nc_mask = (chunk[sample_cols] == "NC")
    sample_nc_counts += nc_mask.sum(axis=0)

sample_missing_rate = sample_nc_counts / total_probes

print("Total probes scanned:", total_probes)
print("Missing rate summary (per sample):")
print(sample_missing_rate.describe())

In [ ]:
import pandas as pd
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint0_snp_clean_v2.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

threshold = 0.02
samples_to_drop = sample_missing_rate[sample_missing_rate > threshold].index.tolist()

print("Number of samples to drop:", len(samples_to_drop))
print(samples_to_drop)

pd.Series(samples_to_drop, name="dropped_sample_id").to_csv(
    os.path.join(out_dir, "qc_log_dropped_samples_missingness.csv"), index=False
)
print("Saved dropped-sample log.")

In [ ]:
import pandas as pd
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint0_snp_clean_v2.txt"
meta_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint1_metadata_binary.csv"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

samples_to_drop_set = set(samples_to_drop)

# --- Rebuild genotype file without these sample columns ---
chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint2_snp_sample_filtered.txt")
first_chunk = True

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    cols_to_drop = [c for c in chunk.columns if c in samples_to_drop_set]
    chunk = chunk.drop(columns=cols_to_drop)
    chunk.to_csv(out_path, sep="\t", mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

print("Saved sample-filtered genotype checkpoint:", out_path)

# --- Sync metadata ---
meta_df_clean = pd.read_csv(meta_path)
meta_df_clean["sample_id"] = meta_df_clean["sample_id"].astype(str)

meta_df_filtered = meta_df_clean[~meta_df_clean["sample_id"].isin(samples_to_drop_set)].copy()
print("Metadata shape after dropping:", meta_df_filtered.shape)
# expect 3396 - 48 = 3348

meta_df_filtered.to_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"), index=False)
print("Saved sample-filtered metadata checkpoint.")

In [ ]:
out_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt"

with open(out_path, encoding="utf-8") as f:
    header = f.readline()
    n_cols = len(header.strip().split("\t"))
    n_rows = sum(1 for _ in f)

print("Genotype checkpoint2 shape (rows, cols):", n_rows, n_cols)
# expect 242764 rows, 3349 cols (3348 samples + 1 ID col)

## Step 2 — Probe missingness filter

Computed only on the sample-filtered matrix (Step 1 must run first). For each probe
(row), compute the fraction of the remaining samples with a no-call. Probes above
threshold are dropped.

**Result:** 3,837 probes dropped, threshold = 5% missingness.

In [ ]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt"

chunksize = 20000
probe_missing_rates = []
probe_ids = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    n_samples = len(geno_cols)

    nc_count = (chunk[geno_cols] == "NC").sum(axis=1)
    missing_rate = nc_count / n_samples

    probe_ids.extend(chunk[id_col].tolist())
    probe_missing_rates.extend(missing_rate.tolist())

probe_missing_series = pd.Series(probe_missing_rates, index=probe_ids)

print("Total probes:", len(probe_missing_series))
print(probe_missing_series.describe())

In [ ]:
import pandas as pd
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

threshold = 0.05
probes_to_drop = set(probe_missing_series[probe_missing_series > threshold].index)

print("Number of probes to drop:", len(probes_to_drop))

pd.Series(list(probes_to_drop), name="dropped_probe_id").to_csv(
    os.path.join(out_dir, "qc_log_dropped_probes_missingness.csv"), index=False
)
print("Saved dropped-probe log.")

In [ ]:
chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint3_snp_probe_filtered.txt")
first_chunk = True

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    chunk = chunk[~chunk[id_col].isin(probes_to_drop)]
    chunk.to_csv(out_path, sep="\t", mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

print("Saved probe-filtered genotype checkpoint:", out_path)

In [ ]:
out_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"

with open(out_path, encoding="utf-8") as f:
    header = f.readline()
    n_cols = len(header.strip().split("\t"))
    n_rows = sum(1 for _ in f)

print("Genotype checkpoint3 shape (rows, cols):", n_rows, n_cols)
# expect 238927 rows, 3349 cols

## Step 3 — Genotype counts + Hardy-Weinberg Equilibrium (HWE)

For each probe, count homozygous-major / heterozygous / homozygous-minor genotypes
across all samples, then run the exact HWE test (Wigginton et al. 2005). **Flagged
only, not filtered** — per supervisor instruction. Flagged list saved to a QC log
for transparency; these probes remain in the analysis.

In [ ]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

chunksize = 20000
alleles = ['A', 'C', 'G', 'T']

results = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    is_nc = (arr == 'NC')
    is_hom = (first == second) & ~is_nc
    is_het = (first != second) & ~is_nc

    n_het = is_het.sum(axis=1)
    n_missing = is_nc.sum(axis=1)

    hom_counts = np.zeros((arr.shape[0], 4), dtype=int)
    for i, letter in enumerate(alleles):
        hom_counts[:, i] = ((first == letter) & is_hom).sum(axis=1)

    sorted_hom = np.sort(hom_counts, axis=1)[:, ::-1]
    n_hom1 = sorted_hom[:, 0]
    n_hom2 = sorted_hom[:, 1]

    for j in range(len(ids)):
        results.append({
            "probe_id": ids[j],
            "n_hom1": n_hom1[j],
            "n_het": n_het[j],
            "n_hom2": n_hom2[j],
            "n_missing": n_missing[j]
        })

counts_df = pd.DataFrame(results)
print(counts_df.shape)
print(counts_df.head())

counts_df.to_csv(os.path.join(out_dir, "checkpoint4_genotype_counts.csv"), index=False)
print("Saved genotype counts checkpoint.")

In [ ]:
import numpy as np
from scipy.special import gammaln
import pandas as pd
from tqdm import tqdm

def hwe_exact_pvalue(n_hom1, n_het, n_hom2):
    """
    Wigginton et al. (2005) exact HWE test.
    Returns two-sided exact p-value.
    """
    n_hom_rare = min(n_hom1, n_hom2)
    n_rare_alleles = 2 * n_hom_rare + n_het
    N = n_hom1 + n_het + n_hom2

    if n_rare_alleles == 0:
        return 1.0

    hets = np.arange(n_rare_alleles % 2, n_rare_alleles + 1, 2)
    homr = (n_rare_alleles - hets) // 2
    homc = N - hets - homr
    valid = homc >= 0
    hets, homr, homc = hets[valid], homr[valid], homc[valid]

    log_liks = (gammaln(N + 1) - gammaln(homr + 1) - gammaln(hets + 1) - gammaln(homc + 1)
                + hets * np.log(2))
    liks = np.exp(log_liks - log_liks.max())
    liks /= liks.sum()

    obs_idx = np.where(hets == n_het)[0]
    if len(obs_idx) == 0:
        return np.nan
    obs_lik = liks[obs_idx[0]]
    return min(liks[liks <= obs_lik + 1e-10].sum(), 1.0)

counts_df = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint4_genotype_counts.csv")

pvals = np.empty(len(counts_df))
for i, row in enumerate(tqdm(counts_df.itertuples(index=False), total=len(counts_df))):
    pvals[i] = hwe_exact_pvalue(row.n_hom1, row.n_het, row.n_hom2)

counts_df["hwe_pvalue"] = pvals
counts_df.to_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint5_hwe_results.csv", index=False)
print("Done. Saved HWE results.")
print(counts_df["hwe_pvalue"].describe())

In [ ]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
threshold = 1e-6

flagged_probes = counts_df.loc[counts_df["hwe_pvalue"] < threshold, ["probe_id", "hwe_pvalue"]]
print("Number of probes flagged for HWE deviation:", len(flagged_probes))

flagged_probes.to_csv(os.path.join(out_dir, "qc_log_hwe_flagged_probes.csv"), index=False)
print("Saved HWE flag log (no probes dropped from data).")

## Step 4 — Overall missingness sanity check

Confirms total remaining NC rate after Steps 1-2, before imputation.

In [ ]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"

chunksize = 20000
total_nc = 0
total_cells = 0

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    geno_cols = chunk.columns[1:]
    total_nc += (chunk[geno_cols] == "NC").sum().sum()
    total_cells += chunk[geno_cols].size

overall_missing_rate = total_nc / total_cells
print("Total NC cells:", total_nc)
print("Total cells:", total_cells)
print("Overall missing rate:", overall_missing_rate)

## Step 5 — Mode imputation

Remaining NC calls (small, sub-1%) filled with the most common observed genotype
at that probe. Runs *before* major-allele computation (Step 6) — since imputed
values are already each probe's mode, this doesn't meaningfully bias the
major-allele call.

In [ ]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint6_snp_imputed.txt")
first_chunk = True

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]

    arr = chunk[geno_cols].values

    for i in range(arr.shape[0]):
        row = arr[i]
        nc_mask = (row == "NC")
        if nc_mask.any():
            valid = row[~nc_mask]
            if len(valid) == 0:
                continue
            vals, counts = np.unique(valid, return_counts=True)
            mode_val = vals[np.argmax(counts)]
            row[nc_mask] = mode_val

    chunk[geno_cols] = arr
    chunk.to_csv(out_path, sep="\t", mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

print("Saved imputed checkpoint:", out_path)

In [ ]:
out_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"

chunksize = 20000
total_nc_remaining = 0
total_rows = 0
n_cols = None

reader = pd.read_csv(out_path, sep="\t", chunksize=chunksize)
for chunk in reader:
    geno_cols = chunk.columns[1:]
    if n_cols is None:
        n_cols = len(chunk.columns)
    total_nc_remaining += (chunk[geno_cols] == "NC").sum().sum()
    total_rows += len(chunk)

print("Total rows:", total_rows)
print("Total cols:", n_cols)
print("Remaining NC count:", total_nc_remaining)

## Step 6 — Major allele computation & 0/1/2 encoding

**See top-of-notebook note on why this replaces manifest-strand-based encoding.**
Major allele computed directly from observed genotype calls (post-imputation) per
probe; genotypes encoded as 0 (hom major) / 1 (het) / 2 (hom minor) relative to it.

**Known pandas-version quirk:** `.loc[ids].values` on the major-allele lookup can
return an Arrow-backed array on newer pandas that breaks `[:, None]` indexing.
If you hit `IndexError: too many indices for array`, change `.values` to
`.to_numpy(dtype=object)` in the ref_alleles line below.

In [ ]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

chunksize = 20000
major_alleles = []
probe_ids_all = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_ids_all.extend(ids.tolist())

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    for i in range(arr.shape[0]):
        all_alleles = np.concatenate([first[i], second[i]])
        vals, counts = np.unique(all_alleles, return_counts=True)
        major = vals[np.argmax(counts)]
        major_alleles.append(major)

major_df = pd.DataFrame({"probe_id": probe_ids_all, "major_allele": major_alleles})
print(major_df.shape)
print(major_df.head())

major_df.to_csv(os.path.join(out_dir, "probe_major_alleles.csv"), index=False)
print("Saved major allele lookup.")

In [ ]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

major_lookup = major_df.set_index("probe_id")["major_allele"]

chunksize = 20000
encoded_chunks = []
probe_order = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_order.extend(ids.tolist())

    ref_alleles = major_lookup.loc[ids].to_numpy(dtype=object)  # fixed: was .values

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    ref_col = ref_alleles[:, None]
    match_first = (first == ref_col)
    match_second = (second == ref_col)
    n_ref_matches = match_first.astype(np.int8) + match_second.astype(np.int8)
    encoded = 2 - n_ref_matches

    encoded_chunks.append(encoded.astype(np.int8))

encoded_matrix = np.vstack(encoded_chunks)
print("Encoded matrix shape:", encoded_matrix.shape)
print("Value counts overall:", np.unique(encoded_matrix, return_counts=True))

In [ ]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

encoded_df = pd.DataFrame(encoded_matrix, columns=[c for c in pd.read_csv(geno_path, sep="\t", nrows=0).columns[1:]])
encoded_df.insert(0, "probe_id", probe_order)

print(encoded_df.shape)
print(encoded_df.head())

encoded_df.to_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"), index=False)
print("Saved encoded SNP matrix (probes as rows, samples as columns).")

## Step 7 — Transpose & merge with metadata

Genotype matrix transposed to samples x probes, joined with sample metadata on
sample_id. Saved as checkpoint8 (pre-relatedness-filter version).

**Note:** confounder PCA, sex-chromosome exclusion, and MAF-in-subsample checks
are handled in `02_confounders.ipynb`, not here — this notebook's output is
genotype-only, phenotype-agnostic.

In [ ]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))

encoded_T = encoded_df.set_index("probe_id").T
encoded_T.index.name = "sample_id"
encoded_T = encoded_T.reset_index()

print("Transposed shape:", encoded_T.shape)
print(encoded_T.iloc[:5, :6])

In [ ]:
print(encoded_T.dtypes.value_counts())
print("Estimated memory (MB):", encoded_T.memory_usage(deep=False).sum() / 1e6)

In [ ]:
import numpy as np

snp_cols_T = encoded_T.columns[1:]
encoded_T[snp_cols_T] = encoded_T[snp_cols_T].astype(np.int8)

print(encoded_T.dtypes.value_counts())
print("Estimated memory (MB):", encoded_T.memory_usage(deep=False).sum() / 1e6)

In [ ]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

meta_df_final = pd.read_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"))
meta_df_final["sample_id"] = meta_df_final["sample_id"].astype(str)

print("Metadata samples:", meta_df_final.shape[0])
print("Genotype samples:", encoded_T.shape[0])
print("Overlap:", len(set(meta_df_final["sample_id"]) & set(encoded_T["sample_id"])))

meta_indexed = meta_df_final.set_index("sample_id")
geno_indexed = encoded_T.set_index("sample_id")

final_df = meta_indexed.join(geno_indexed, how="inner")
final_df = final_df.reset_index()

print("Final merged shape:", final_df.shape)
print(final_df.iloc[:5, :12])

In [ ]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

final_df.to_csv(os.path.join(out_dir, "checkpoint8_final_features_targets.csv"), index=False)
print("Saved final merged dataset:", os.path.join(out_dir, "checkpoint8_final_features_targets.csv"))
print("Shape:", final_df.shape)

## Step 8 — Relatedness / duplicate sample filtering

**Why this step exists:** All downstream statistical methods (DoubleML residualization,
stability selection, the PC algorithm) assume each sample is an independent observation.
This assumption was not checked in the original pipeline. It was discovered via an
unrelated diagnostic — the genomic inflation factor (lambda) computed in
`02_confounders.ipynb` came back abnormally high (lambda ~1.7-1.8 at 10 PCs), and adding
more PCs made it *worse*, which is the signature of relatedness contamination rather
than uncorrected population structure.

**Method:** Compute pairwise genotype correlation across a random 5,000-SNP subset.
Samples with correlation > 0.5 to any other sample are treated as duplicates or close
relatives. Build a graph where nodes are samples and edges connect correlated pairs;
find connected components (clusters); retain one representative per cluster (currently:
lowest sample_id — a simple deterministic tie-break).

**Threshold justification:** 0.5 is a conservative, high-confidence threshold, chosen
to remove near-certain duplicates and close relatives while avoiding the more
ambiguous 0.3-0.5 band, which is harder to distinguish from legitimate population
admixture correlation (a known, separate, well-documented phenomenon in
African-American genetic data — see PC1-gender finding in `02_confounders.ipynb`).

**Result:**
- Pairs found above threshold 0.5: 375
- Related/duplicate clusters: 249
- Total samples involved: 561
- Samples dropped: 312
- Samples remaining: 3,036 (from 3,348)

**Interpretation:** ~9.3% of the AA cohort consisted of duplicate or closely related
individuals — proportionally similar to the EA cohort (8.5%), suggesting this is a
consistent data-collection artifact affecting both GEO datasets rather than something
specific to one cohort. This step is mandatory going forward — every downstream
notebook depends on `checkpoint7b`/`checkpoint2b`, not the original `checkpoint7`/`checkpoint2`.

**Outputs:**
- `checkpoint7b_snp_encoded_012_relatedness_filtered.csv`
- `checkpoint2b_metadata_relatedness_filtered.csv`
- `qc_log_dropped_relatedness.csv`

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# --- load QC'd genotype matrix (output of Step 7) ---
encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

# --- exclude sex-linked probes (relatedness should be estimated from autosomal data) ---
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

# --- compute MAF/standardization params without building full float64 matrix ---
X_f32 = X_auto_int.astype(np.float32)
p_aa = X_f32.mean(axis=1) / 2
denom_aa = np.sqrt(2 * p_aa * (1 - p_aa))
valid_mask = denom_aa > 1e-8
del X_f32
gc.collect()

# --- random 5000-SNP subset for relatedness estimation ---
valid_indices = np.where(valid_mask)[0]
np.random.seed(0)
chosen_idx = np.random.RandomState(0).choice(valid_indices, 5000, replace=False)

X_subset_int = X_auto_int[chosen_idx].astype(np.float64)
del X_auto_int
gc.collect()

p_subset = p_aa[chosen_idx]
denom_subset = denom_aa[chosen_idx]
X_subset_std = ((X_subset_int - 2 * p_subset[:, None]) / denom_subset[:, None]).T
del X_subset_int
gc.collect()

# --- pairwise correlation ---
sample_corr = np.corrcoef(X_subset_std)
del X_subset_std
gc.collect()
np.fill_diagonal(sample_corr, 0)

print("Correlation matrix shape:", sample_corr.shape)

# --- build relatedness graph, threshold 0.5 ---
threshold = 0.5
iu = np.triu_indices_from(sample_corr, k=1)
rows, cols = iu
strong = sample_corr[iu] > threshold

pairs = [(sample_ids[rows[k]], sample_ids[cols[k]], sample_corr[iu][k])
         for k in range(len(strong)) if strong[k]]
print(f"Pairs above threshold {threshold}: {len(pairs)}")

G = nx.Graph()
G.add_nodes_from(sample_ids)
for a, b, r in pairs:
    G.add_edge(a, b, weight=r)

clusters = [c for c in nx.connected_components(G) if len(c) > 1]
print(f"Related clusters: {len(clusters)}")
print(f"Total samples involved: {sum(len(c) for c in clusters)}")

samples_to_drop = []
for cluster in clusters:
    sorted_cluster = sorted(cluster)
    samples_to_drop.extend(sorted_cluster[1:])  # keep first, drop rest

print(f"\nSamples to drop: {len(samples_to_drop)}")
print(f"Samples remaining: {len(sample_ids) - len(samples_to_drop)}")

pd.Series(samples_to_drop, name="dropped_sample_id_relatedness").to_csv(
    os.path.join(out_dir, "qc_log_dropped_relatedness.csv"), index=False
)
print("Saved drop log.")

In [ ]:
# --- apply the drop list: rebuild metadata and genotype checkpoint files ---
samples_to_drop_set = set(samples_to_drop)

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df_filtered = meta_df[~meta_df["sample_id"].isin(samples_to_drop_set)].copy()
print("Metadata:", meta_df.shape, "->", meta_df_filtered.shape)
meta_df_filtered.to_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"), index=False)

geno_path = os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv")
out_path = os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv")
chunksize = 20000
first_chunk = True
reader = pd.read_csv(geno_path, chunksize=chunksize)
for chunk in reader:
    cols_to_keep = [c for c in chunk.columns if c == "probe_id" or c not in samples_to_drop_set]
    chunk = chunk[cols_to_keep]
    chunk.to_csv(out_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
    first_chunk = False

check_df = pd.read_csv(out_path, nrows=0)
print("Genotype samples remaining:", len(check_df.columns) - 1)
print("\nStep 8 complete. 01_qc.ipynb finished.")